## Imports and config

In [1]:
from pathlib import Path
import os
import pandas as pd
import numpy as np

In [21]:
OLD_CSV = "/root/workspace/PlantCLEF2026/src_experiments/002_bioclip_tile_zero_shot_v2/data/PlantCLEF2024_single_plant_training_metadata.csv"

NEW_CSV = "/workspace/plantclef/processed/inat_research_grade_manifest.csv"

OLD_IMG_ROOT = Path("/workspace/plantclef/raw/train/images_max_side_800")

OUT_DIR = Path("/root/workspace/PlantCLEF2026/src_experiments/i001_data_download/data/from_scratch")
OUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_IMAGES_PER_SPECIES = 100
MAX_IMAGES_PER_SPECIES = 500
ALPHA = 0.5
SEED = 26

## Read old and new data

In [22]:
old_raw = pd.read_csv(OLD_CSV, sep=";", low_memory=False)
new_raw = pd.read_csv(NEW_CSV, low_memory=False)

print("Old rows:", len(old_raw))
print("New rows:", len(new_raw))

print("\nOld columns:")
print(old_raw.columns.tolist())

print("\nNew columns:")
print(new_raw.columns.tolist())

Old rows: 1408033
New rows: 1338626

Old columns:
['image_name', 'organ', 'species_id', 'obs_id', 'license', 'partner', 'author', 'altitude', 'latitude', 'longitude', 'gbif_species_id', 'species', 'genus', 'family', 'dataset', 'publisher', 'references', 'url', 'learn_tag', 'image_backup_url']

New columns:
['image_path', 'species_id', 'gbif_species_id', 'gbif_occurrence_id', 'license', 'scientific_name', 'url']


## Clean new data by `image_path`

In [23]:
new_clean = new_raw.copy()

before = len(new_clean)
new_clean = new_clean.drop_duplicates(subset=["image_path"]).copy()
after = len(new_clean)

print("New rows before image_path dedupe:", before)
print("New rows after image_path dedupe:", after)
print("Removed duplicate image_path rows:", before - after)

new_clean.head()

New rows before image_path dedupe: 1338626
New rows after image_path dedupe: 1245748
Removed duplicate image_path rows: 92878


,image_path,species_id,gbif_species_id,gbif_occurrence_id,license,scientific_name,url
0,/workspace/plantclef/raw/inat_research_grade/1...,1396710,5284517,5938299218,http://creativecommons.org/licenses/by-nc/4.0/...,Taxus baccata L.,https://inaturalist-open-data.s3.amazonaws.com...
1,/workspace/plantclef/raw/inat_research_grade/1...,1396710,5284517,5938200063,http://creativecommons.org/licenses/by-nc/4.0/...,Taxus baccata L.,https://inaturalist-open-data.s3.amazonaws.com...
2,/workspace/plantclef/raw/inat_research_grade/1...,1396710,5284517,5938168811,http://creativecommons.org/publicdomain/zero/1...,Taxus baccata L.,https://inaturalist-open-data.s3.amazonaws.com...
3,/workspace/plantclef/raw/inat_research_grade/1...,1396710,5284517,5938071098,http://creativecommons.org/licenses/by/4.0/leg...,Taxus baccata L.,https://inaturalist-open-data.s3.amazonaws.com...
4,/workspace/plantclef/raw/inat_research_grade/1...,1396710,5284517,5938251864,http://creativecommons.org/licenses/by/4.0/leg...,Taxus baccata L.,https://inaturalist-open-data.s3.amazonaws.com...


Save it:

In [24]:
new_clean_out = OUT_DIR / "inat_research_grade_manifest_clean_image_path.csv"

new_clean.to_csv(new_clean_out, index=False)

print("Saved:", new_clean_out)

Saved: /root/workspace/PlantCLEF2026/src_experiments/i001_data_download/data/from_scratch/inat_research_grade_manifest_clean_image_path.csv


## Build old image paths

In [25]:
old = old_raw.copy()

old["species_id"] = pd.to_numeric(old["species_id"], errors="coerce").astype("Int64")
old["gbif_species_id"] = pd.to_numeric(old["gbif_species_id"], errors="coerce").astype("Int64")

old["image_path"] = old.apply(
    lambda row: str(
        OLD_IMG_ROOT
        / str(row["species_id"])
        / str(row["image_name"])
    ),
    axis=1,
)

old[["species_id", "image_name", "image_path"]].head()

,species_id,image_name,image_path
0,1396710,59feabe1c98f06e7f819f73c8246bd8f1a89556b.jpg,/workspace/plantclef/raw/train/images_max_side...
1,1396710,dc273995a89827437d447f29a52ccac86f65476e.jpg,/workspace/plantclef/raw/train/images_max_side...
2,1396710,416235e7023a4bd1513edf036b6097efc693a304.jpg,/workspace/plantclef/raw/train/images_max_side...
3,1396710,cbd18fade82c46a5c725f1f3d982174895158afc.jpg,/workspace/plantclef/raw/train/images_max_side...
4,1396710,f82c8c6d570287ebed8407cefcfcb2a51eaaf56e.jpg,/workspace/plantclef/raw/train/images_max_side...


In [26]:
example_old_path = old["image_path"].iloc[0]

print(example_old_path)
print("Exists:", os.path.exists(example_old_path))

/workspace/plantclef/raw/train/images_max_side_800/1396710/59feabe1c98f06e7f819f73c8246bd8f1a89556b.jpg
Exists: True


## Standardize old data

In [27]:
old_std = pd.DataFrame({
    "image_path": old["image_path"],
    "image_name": old["image_name"],
    "species_id": old["species_id"],
    "gbif_species_id": old["gbif_species_id"],
    "scientific_name": old["species"],
    "license": old["license"],
    "source": "old_plantclef",
})

if "genus" in old.columns:
    old_std["genus"] = old["genus"]

if "family" in old.columns:
    old_std["family"] = old["family"]

if "url" in old.columns:
    old_std["url"] = old["url"]

old_std.head()

,image_path,image_name,species_id,gbif_species_id,scientific_name,license,source,genus,family,url
0,/workspace/plantclef/raw/train/images_max_side...,59feabe1c98f06e7f819f73c8246bd8f1a89556b.jpg,1396710,5284517,Taxus baccata L.,cc-by-sa,old_plantclef,Taxus,Taxaceae,https://bs.plantnet.org/image/o/59feabe1c98f06...
1,/workspace/plantclef/raw/train/images_max_side...,dc273995a89827437d447f29a52ccac86f65476e.jpg,1396710,5284517,Taxus baccata L.,cc-by-sa,old_plantclef,Taxus,Taxaceae,https://bs.plantnet.org/image/o/dc273995a89827...
2,/workspace/plantclef/raw/train/images_max_side...,416235e7023a4bd1513edf036b6097efc693a304.jpg,1396710,5284517,Taxus baccata L.,cc-by-sa,old_plantclef,Taxus,Taxaceae,https://bs.plantnet.org/image/o/416235e7023a4b...
3,/workspace/plantclef/raw/train/images_max_side...,cbd18fade82c46a5c725f1f3d982174895158afc.jpg,1396710,5284517,Taxus baccata L.,cc-by-sa,old_plantclef,Taxus,Taxaceae,https://bs.plantnet.org/image/o/cbd18fade82c46...
4,/workspace/plantclef/raw/train/images_max_side...,f82c8c6d570287ebed8407cefcfcb2a51eaaf56e.jpg,1396710,5284517,Taxus baccata L.,cc-by-sa,old_plantclef,Taxus,Taxaceae,https://bs.plantnet.org/image/o/f82c8c6d570287...


## Standardize new data

In [28]:
new = new_clean.copy()

new["species_id"] = pd.to_numeric(new["species_id"], errors="coerce").astype("Int64")
new["gbif_species_id"] = pd.to_numeric(new["gbif_species_id"], errors="coerce").astype("Int64")

new_std = pd.DataFrame({
    "image_path": new["image_path"],
    "image_name": new["image_path"].astype(str).apply(lambda x: Path(x).name),
    "species_id": new["species_id"],
    "gbif_species_id": new["gbif_species_id"],
    "scientific_name": new["scientific_name"],
    "license": new["license"],
    "source": "new_inat",
})

if "url" in new.columns:
    new_std["url"] = new["url"]

if "gbif_occurrence_id" in new.columns:
    new_std["gbif_occurrence_id"] = new["gbif_occurrence_id"]

new_std.head()

,image_path,image_name,species_id,gbif_species_id,scientific_name,license,source,url,gbif_occurrence_id
0,/workspace/plantclef/raw/inat_research_grade/1...,5938299218_605069874.jpg,1396710,5284517,Taxus baccata L.,http://creativecommons.org/licenses/by-nc/4.0/...,new_inat,https://inaturalist-open-data.s3.amazonaws.com...,5938299218
1,/workspace/plantclef/raw/inat_research_grade/1...,5938200063_605359819.jpg,1396710,5284517,Taxus baccata L.,http://creativecommons.org/licenses/by-nc/4.0/...,new_inat,https://inaturalist-open-data.s3.amazonaws.com...,5938200063
2,/workspace/plantclef/raw/inat_research_grade/1...,5938168811_604750178.jpg,1396710,5284517,Taxus baccata L.,http://creativecommons.org/publicdomain/zero/1...,new_inat,https://inaturalist-open-data.s3.amazonaws.com...,5938168811
3,/workspace/plantclef/raw/inat_research_grade/1...,5938071098_605001704.jpg,1396710,5284517,Taxus baccata L.,http://creativecommons.org/licenses/by/4.0/leg...,new_inat,https://inaturalist-open-data.s3.amazonaws.com...,5938071098
4,/workspace/plantclef/raw/inat_research_grade/1...,5938251864_604497369.jpg,1396710,5284517,Taxus baccata L.,http://creativecommons.org/licenses/by/4.0/leg...,new_inat,https://inaturalist-open-data.s3.amazonaws.com...,5938251864


## Combine old + new

In [29]:
combined = pd.concat([old_std, new_std], ignore_index=True)

combined = combined.dropna(subset=["species_id", "image_path"]).copy()

combined["species_id"] = pd.to_numeric(combined["species_id"], errors="coerce").astype("Int64")
combined["gbif_species_id"] = pd.to_numeric(combined["gbif_species_id"], errors="coerce").astype("Int64")

before = len(combined)
combined = combined.drop_duplicates(subset=["image_path"]).copy()
after = len(combined)

print("Combined rows:", len(combined))
print("Combined species:", combined["species_id"].nunique())
print("Removed duplicate image_path rows:", before - after)

combined.head()

Combined rows: 2653781
Combined species: 7806
Removed duplicate image_path rows: 0


,image_path,image_name,species_id,gbif_species_id,scientific_name,license,source,genus,family,url,gbif_occurrence_id
0,/workspace/plantclef/raw/train/images_max_side...,59feabe1c98f06e7f819f73c8246bd8f1a89556b.jpg,1396710,5284517,Taxus baccata L.,cc-by-sa,old_plantclef,Taxus,Taxaceae,https://bs.plantnet.org/image/o/59feabe1c98f06...,NaN
1,/workspace/plantclef/raw/train/images_max_side...,dc273995a89827437d447f29a52ccac86f65476e.jpg,1396710,5284517,Taxus baccata L.,cc-by-sa,old_plantclef,Taxus,Taxaceae,https://bs.plantnet.org/image/o/dc273995a89827...,NaN
2,/workspace/plantclef/raw/train/images_max_side...,416235e7023a4bd1513edf036b6097efc693a304.jpg,1396710,5284517,Taxus baccata L.,cc-by-sa,old_plantclef,Taxus,Taxaceae,https://bs.plantnet.org/image/o/416235e7023a4b...,NaN
3,/workspace/plantclef/raw/train/images_max_side...,cbd18fade82c46a5c725f1f3d982174895158afc.jpg,1396710,5284517,Taxus baccata L.,cc-by-sa,old_plantclef,Taxus,Taxaceae,https://bs.plantnet.org/image/o/cbd18fade82c46...,NaN
4,/workspace/plantclef/raw/train/images_max_side...,f82c8c6d570287ebed8407cefcfcb2a51eaaf56e.jpg,1396710,5284517,Taxus baccata L.,cc-by-sa,old_plantclef,Taxus,Taxaceae,https://bs.plantnet.org/image/o/f82c8c6d570287...,NaN


Save full combined manifest:

In [30]:
combined_full_out = OUT_DIR / "combined_old_new_manifest_full.csv"

combined.to_csv(combined_full_out, index=False)

print("Saved:", combined_full_out)

Saved: /root/workspace/PlantCLEF2026/src_experiments/i001_data_download/data/from_scratch/combined_old_new_manifest_full.csv


## Species distribution before capping

In [31]:
combined_counts = (
    combined.groupby("species_id")
    .agg(
        n_images=("image_path", "count"),
        n_old=("source", lambda x: int((x == "old_plantclef").sum())),
        n_new=("source", lambda x: int((x == "new_inat").sum())),
        scientific_name=("scientific_name", "first"),
        gbif_species_id=("gbif_species_id", "first"),
    )
    .reset_index()
    .sort_values("n_images", ascending=True)
)

combined_counts.head(50)

,species_id,n_images,n_old,n_new,scientific_name,gbif_species_id
7756,1744551,1,1,0,"Helianthemum dianicum Pérez Dacosta, M.B.Cresp...",8418210
7743,1744511,1,1,0,Genista tribracteolata (Webb) Pau,5354550
5806,1457840,1,1,0,Ferulago brachyloba Boiss. & Reut.,3635847
6540,1647166,1,1,0,Armeria muelleri A.Huet,5668284
7805,1744937,1,1,0,"Linaria semialata D.López, Sánchez-Gómez, J.F....",11164302
7731,1744480,1,1,0,Armeria cantabrica Boiss. & Reut. ex Willk.,7298203
3782,1392852,1,1,0,Hieracium chaixianum Arv.-Touv. & Gaut.,4229027
6532,1647141,1,1,0,Armeria castellana Boiss. & Reut. ex Leresche,7683925
3959,1393553,1,1,0,Limonium corsicum Erben,4088422
6503,1646226,1,1,0,Linaria bubanii Font Quer,8423889


## Count bins

In [32]:
bins = [0, 1, 2, 5, 10, 20, 50, 100, 250, 500, 1000, 2000, 5000, 10_000_000]

labels = [
    "1",
    "2",
    "3-5",
    "6-10",
    "11-20",
    "21-50",
    "51-100",
    "101-250",
    "251-500",
    "501-1000",
    "1001-2000",
    "2001-5000",
    ">5000",
]

combined_counts["count_bin"] = pd.cut(
    combined_counts["n_images"],
    bins=bins,
    labels=labels,
    include_lowest=True,
)

combined_counts["count_bin"].value_counts().sort_index()

count_bin
1             145
2             109
3-5           344
6-10          381
11-20         437
21-50         701
51-100        584
101-250      1217
251-500      1175
501-1000     2596
1001-2000     117
2001-5000       0
>5000           0
Name: count, dtype: int64

## Get species with fewer than 100 images

In [33]:
under100_species = combined_counts[
    combined_counts["n_images"] < TARGET_IMAGES_PER_SPECIES
].copy()

under100_species["needed_to_100"] = TARGET_IMAGES_PER_SPECIES - under100_species["n_images"]

under100_species = under100_species.sort_values(
    ["n_images", "needed_to_100"],
    ascending=[True, False],
)

print("Species with fewer than 100 images:", len(under100_species))
print("Total extra images needed to reach 100 each:", int(under100_species["needed_to_100"].sum()))

under100_species[
    [
        "species_id",
        "scientific_name",
        "gbif_species_id",
        "n_images",
        "n_old",
        "n_new",
        "needed_to_100",
    ]
].head(100)

Species with fewer than 100 images: 2695
Total extra images needed to reach 100 each: 192341


,species_id,scientific_name,gbif_species_id,n_images,n_old,n_new,needed_to_100
7756,1744551,"Helianthemum dianicum Pérez Dacosta, M.B.Cresp...",8418210,1,1,0,99
7743,1744511,Genista tribracteolata (Webb) Pau,5354550,1,1,0,99
5806,1457840,Ferulago brachyloba Boiss. & Reut.,3635847,1,1,0,99
6540,1647166,Armeria muelleri A.Huet,5668284,1,1,0,99
7805,1744937,"Linaria semialata D.López, Sánchez-Gómez, J.F....",11164302,1,1,0,99
...,...,...,...,...,...,...,...
7070,1738690,Verbascum × kerneri Borbás,12234790,1,1,0,99
7229,1742054,Aster bellidiastrum (L.) Scop.,11072581,1,1,0,99
7244,1742165,Rubus pulcher P.J.Müll. & Lefèvre,2992874,1,1,0,99
5919,1489066,Campanula × chevalieri Sennen,5411337,1,1,0,99


In [34]:
under100_out = OUT_DIR / "species_under100_collection_targets.csv"

under100_species.to_csv(under100_out, index=False)

print("Saved:", under100_out)

Saved: /root/workspace/PlantCLEF2026/src_experiments/i001_data_download/data/from_scratch/species_under100_collection_targets.csv


## Cap very large species

In [39]:
def cap_per_species(df, max_images_per_species=500, seed=42):
    if max_images_per_species <= 0:
        return df.copy()

    df = df.copy()

    # If species_id is accidentally in the index, bring it back as a column
    if "species_id" not in df.columns:
        df = df.reset_index()

    capped = (
        df.sample(frac=1, random_state=seed)   # shuffle first
        .groupby("species_id", group_keys=False)
        .head(max_images_per_species)
        .reset_index(drop=True)
    )

    return capped

In [40]:
combined_capped = cap_per_species(
    combined,
    max_images_per_species=MAX_IMAGES_PER_SPECIES,
    seed=SEED,
)

print("Before cap:", len(combined))
print("After cap:", len(combined_capped))
print("Species:", combined_capped["species_id"].nunique())

Before cap: 2653781
After cap: 2075453
Species: 7806


Check cap worked:

In [42]:
combined_capped.groupby("species_id").size().describe(
    percentiles=[0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
)

count    7806.000000
mean      265.879195
std       206.468857
min         1.000000
1%          1.000000
5%          4.000000
10%         7.000000
25%        41.250000
50%       249.000000
75%       500.000000
90%       500.000000
95%       500.000000
99%       500.000000
max       500.000000
dtype: float64

## Add sampling weights

In [43]:
def add_sampling_weights(df, label_col="species_id", alpha=0.5):
    """
    alpha = 0.0  -> raw sampling
    alpha = 0.5  -> sqrt-balanced sampling
    alpha = 1.0  -> fully class-balanced sampling
    """
    df = df.copy()

    class_counts = df[label_col].value_counts().to_dict()

    df["class_count"] = df[label_col].map(class_counts).astype(int)
    df["sample_weight"] = 1.0 / (df["class_count"] ** alpha)

    return df

In [44]:
combined_weighted = add_sampling_weights(
    combined_capped,
    label_col="species_id",
    alpha=ALPHA,
)

combined_weighted.head()

,image_path,image_name,species_id,gbif_species_id,scientific_name,license,source,genus,family,url,gbif_occurrence_id,class_count,sample_weight
0,/workspace/plantclef/raw/train/images_max_side...,c962b372465d61d41e2eb4606e44b933ef9802b5.jpg,1397476,2889010,Rumex cristatus DC.,http://creativecommons.org/licenses/by-nc/4.0/,old_plantclef,Rumex,Polygonaceae,https://inaturalist-open-data.s3.amazonaws.com...,NaN,500,0.044721
1,/workspace/plantclef/raw/train/images_max_side...,15784287b7f8b786f34d065faa635bfc9947a074.jpg,1396486,3151618,Symphyotrichum novae-angliae (L.) G.L.Nesom,cc-by-sa,old_plantclef,Symphyotrichum,Asteraceae,https://bs.plantnet.org/image/o/15784287b7f8b7...,NaN,500,0.044721
2,/workspace/plantclef/raw/inat_research_grade/1...,5154400391_318234752.jpg,1397517,2951998,Ulex gallii Planch.,http://creativecommons.org/licenses/by-nc/4.0/...,new_inat,NaN,NaN,https://inaturalist-open-data.s3.amazonaws.com...,5.154400e+09,500,0.044721
3,/workspace/plantclef/raw/inat_research_grade/1...,4075609191_262446925.jpg,1722478,3053061,Matthiola parviflora (Schousb.) W.T.Aiton,http://creativecommons.org/licenses/by-nc/4.0/...,new_inat,NaN,NaN,https://inaturalist-open-data.s3.amazonaws.com...,4.075609e+09,500,0.044721
4,/workspace/plantclef/raw/train/images_max_side...,1abb387c5369ee36aeb8a960fd0384e3b3c1bd21.jpg,1363200,5568629,Anabasis articulata (Forssk.) Moq.,cc-by-sa,old_plantclef,Anabasis,Amaranthaceae,https://bs.plantnet.org/image/o/1abb387c5369ee...,NaN,347,0.053683


## Save final training manifest

In [45]:
suffix = f"capped{MAX_IMAGES_PER_SPECIES}_alpha{str(ALPHA).replace('.', '')}"

weighted_out = OUT_DIR / f"combined_old_new_manifest_{suffix}_weights.csv"

combined_weighted.to_csv(weighted_out, index=False)

print("Saved:", weighted_out)
print("Rows:", len(combined_weighted))
print("Species:", combined_weighted["species_id"].nunique())

Saved: /root/workspace/PlantCLEF2026/src_experiments/i001_data_download/data/from_scratch/combined_old_new_manifest_capped500_alpha05_weights.csv
Rows: 2075453
Species: 7806


## Final species summary after capping

In [46]:
species_summary_final = (
    combined_weighted.groupby("species_id")
    .agg(
        n_images=("image_path", "count"),
        n_old=("source", lambda x: int((x == "old_plantclef").sum())),
        n_new=("source", lambda x: int((x == "new_inat").sum())),
        scientific_name=("scientific_name", "first"),
        gbif_species_id=("gbif_species_id", "first"),
        class_count=("class_count", "first"),
        sample_weight=("sample_weight", "first"),
    )
    .reset_index()
    .sort_values("n_images", ascending=True)
)

species_summary_final.head(100)

,species_id,n_images,n_old,n_new,scientific_name,gbif_species_id,class_count,sample_weight
7702,1744380,1,1,0,Limonium subglabrum Erben,4089427,1,1.0
7731,1744480,1,1,0,Armeria cantabrica Boiss. & Reut. ex Willk.,7298203,1,1.0
5826,1467324,1,1,0,Hedera rhizomatifera (McAll.) Jury,3036082,1,1.0
3782,1392852,1,1,0,Hieracium chaixianum Arv.-Touv. & Gaut.,4229027,1,1.0
3783,1392890,1,1,0,Hieracium dasytrichum Arv.-Touv.,3135152,1,1.0
...,...,...,...,...,...,...,...,...
665,1357866,1,1,0,Senecio petraeus Boiss. & Reut.,7980850,1,1.0
6139,1519429,1,1,0,Schlagintweitia chamaepicris (Arv.-Touv.) Greuter,3115485,1,1.0
3931,1393503,1,1,0,Leucanthemum gaudinii Dalla Torre,5400952,1,1.0
6737,1673405,1,1,0,Galium valentinum Lange,2913386,1,1.0


Save:

In [47]:
species_summary_out = OUT_DIR / f"combined_old_new_species_summary_{suffix}.csv"

species_summary_final.to_csv(species_summary_out, index=False)

print("Saved:", species_summary_out)

Saved: /root/workspace/PlantCLEF2026/src_experiments/i001_data_download/data/from_scratch/combined_old_new_species_summary_capped500_alpha05.csv


## Final check

In [48]:
print("Full combined rows:", len(combined))
print("Capped/weighted rows:", len(combined_weighted))
print("Species:", combined_weighted["species_id"].nunique())

print("Old images in final:", int((combined_weighted["source"] == "old_plantclef").sum()))
print("New images in final:", int((combined_weighted["source"] == "new_inat").sum()))

print("Min images/species:", species_summary_final["n_images"].min())
print("Median images/species:", species_summary_final["n_images"].median())
print("Max images/species:", species_summary_final["n_images"].max())

Full combined rows: 2653781
Capped/weighted rows: 2075453
Species: 7806
Old images in final: 1138411
New images in final: 937042
Min images/species: 1
Median images/species: 249.0
Max images/species: 500


# Get additional data

In [1]:
from pathlib import Path
import os
import time
import json
import hashlib
import requests
import pandas as pd
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

In [2]:
BASE_DIR = Path("/root/workspace/PlantCLEF2026/src_experiments/i001_data_download/data/from_scratch")
OUT_DIR = Path("/root/workspace/PlantCLEF2026/src_experiments/i001_data_download/data/extra_under100")
DOWNLOAD_ROOT = Path("/workspace/plantclef/raw/extra_under100")

OUT_DIR.mkdir(parents=True, exist_ok=True)
DOWNLOAD_ROOT.mkdir(parents=True, exist_ok=True)

# Use the species summary you created from combined old + new data.
SPECIES_SUMMARY_CSV = BASE_DIR / "combined_old_new_species_summary_capped500_alpha05.csv"

TARGET_IMAGES_PER_SPECIES = 100

# Collect extra candidates, not just exactly the missing amount,
# because some URLs may fail or duplicate existing data.
CANDIDATE_MULTIPLIER = 3
MAX_CANDIDATES_PER_SPECIES = 300

# Rate limiting
REQUEST_SLEEP_S = 0.35

# License policy.
# For non-commercial research, NC can be included.
ALLOW_NC = True

if ALLOW_NC:
    INAT_LICENSES = ["cc0", "cc-by", "cc-by-sa", "cc-by-nc", "cc-by-nc-sa"]
else:
    INAT_LICENSES = ["cc0", "cc-by", "cc-by-sa"]

print("Using licenses:", INAT_LICENSES)

Using licenses: ['cc0', 'cc-by', 'cc-by-sa', 'cc-by-nc', 'cc-by-nc-sa']


## Load species summary and select under-100 species

In [4]:
species_summary = pd.read_csv(SPECIES_SUMMARY_CSV)

species_summary.head()

,species_id,n_images,n_old,n_new,scientific_name,gbif_species_id,class_count,sample_weight
0,1744380,1,1,0,Limonium subglabrum Erben,4089427.0,1,1.0
1,1744480,1,1,0,Armeria cantabrica Boiss. & Reut. ex Willk.,7298203.0,1,1.0
2,1467324,1,1,0,Hedera rhizomatifera (McAll.) Jury,3036082.0,1,1.0
3,1392852,1,1,0,Hieracium chaixianum Arv.-Touv. & Gaut.,4229027.0,1,1.0
4,1392890,1,1,0,Hieracium dasytrichum Arv.-Touv.,3135152.0,1,1.0


In [5]:
targets = species_summary[species_summary["n_images"] < TARGET_IMAGES_PER_SPECIES].copy()

targets["needed_to_100"] = TARGET_IMAGES_PER_SPECIES - targets["n_images"]

targets = targets.sort_values(["n_images", "needed_to_100"], ascending=[True, False])

print("Species under 100 images:", len(targets))
print("Total images needed to reach 100:", int(targets["needed_to_100"].sum()))

targets[
    [
        "species_id",
        "scientific_name",
        "gbif_species_id",
        "n_images",
        "n_old",
        "n_new",
        "needed_to_100",
    ]
].head(50)

Species under 100 images: 2695
Total images needed to reach 100: 192341


,species_id,scientific_name,gbif_species_id,n_images,n_old,n_new,needed_to_100
0,1744380,Limonium subglabrum Erben,4089427.0,1,1,0,99
1,1744480,Armeria cantabrica Boiss. & Reut. ex Willk.,7298203.0,1,1,0,99
2,1467324,Hedera rhizomatifera (McAll.) Jury,3036082.0,1,1,0,99
3,1392852,Hieracium chaixianum Arv.-Touv. & Gaut.,4229027.0,1,1,0,99
4,1392890,Hieracium dasytrichum Arv.-Touv.,3135152.0,1,1,0,99
5,1394329,Orobanche bartlingii Griseb.,3734924.0,1,1,0,99
6,1495430,Cistus × stenophyllus Link,3595194.0,1,1,0,99
7,1564351,Teucrium leonis Sennen,3895822.0,1,1,0,99
8,1564457,"Thymus × aitanae Mateo, M.B.Crespo & E.Laguna",5607780.0,1,1,0,99
9,1744937,"Linaria semialata D.López, Sánchez-Gómez, J.F....",11164302.0,1,1,0,99


Save the target list:

In [6]:
targets_out = OUT_DIR / "species_under100_targets.csv"
targets.to_csv(targets_out, index=False)

print("Saved:", targets_out)

Saved: /root/workspace/PlantCLEF2026/src_experiments/i001_data_download/data/extra_under100/species_under100_targets.csv


## iNaturalist API helper

In [7]:
INAT_OBS_URL = "https://api.inaturalist.org/v1/observations"


def normalize_inat_photo_url(url, size="original"):
    if not isinstance(url, str):
        return None

    # iNat URLs often contain square/small/medium/large/original.
    for s in ["square", "small", "medium", "large", "original"]:
        url = url.replace(f"/{s}.", f"/{size}.")

    return url


def query_inat_species(scientific_name, max_photos=100, sleep_s=0.35):
    rows = []
    page = 1
    per_page = 200

    while len(rows) < max_photos:
        params = {
            "taxon_name": scientific_name,
            "quality_grade": "research",
            "has[]": "photos",
            "photo_license": ",".join(INAT_LICENSES),
            "per_page": per_page,
            "page": page,
            "order_by": "created_at",
            "order": "desc",
        }

        try:
            r = requests.get(INAT_OBS_URL, params=params, timeout=30)
        except Exception as e:
            print("iNat request exception:", scientific_name, repr(e))
            break

        if r.status_code != 200:
            print("iNat error:", r.status_code, scientific_name, r.text[:300])
            break

        data = r.json()
        results = data.get("results", [])

        if not results:
            break

        for obs in results:
            obs_id = obs.get("id")
            obs_uri = obs.get("uri")
            taxon = obs.get("taxon") or {}
            photos = obs.get("photos") or []

            for photo in photos:
                raw_url = photo.get("url")
                photo_url = normalize_inat_photo_url(raw_url, size="original")

                if not photo_url:
                    continue

                rows.append({
                    "api_source": "inat",
                    "query_scientific_name": scientific_name,
                    "matched_name": taxon.get("name"),
                    "source_observation_id": obs_id,
                    "source_photo_id": photo.get("id"),
                    "url": photo_url,
                    "raw_url": raw_url,
                    "license": photo.get("license_code"),
                    "attribution": photo.get("attribution"),
                    "reference": obs_uri,
                    "observed_on": obs.get("observed_on"),
                    "quality_grade": obs.get("quality_grade"),
                    "latitude": (
                        obs.get("geojson", {}).get("coordinates", [None, None])[1]
                        if obs.get("geojson") else None
                    ),
                    "longitude": (
                        obs.get("geojson", {}).get("coordinates", [None, None])[0]
                        if obs.get("geojson") else None
                    ),
                })

                if len(rows) >= max_photos:
                    break

            if len(rows) >= max_photos:
                break

        page += 1
        time.sleep(sleep_s)

        total_results = data.get("total_results")
        if total_results is not None and (page - 1) * per_page >= total_results:
            break

    return rows

## GBIF API helper

In [8]:
GBIF_OCC_URL = "https://api.gbif.org/v1/occurrence/search"


def extract_gbif_media_rows(record):
    rows = []

    media_list = record.get("media") or []

    for m in media_list:
        image_url = (
            m.get("identifier")
            or m.get("references")
            or m.get("source")
        )

        if not image_url:
            continue

        rows.append({
            "api_source": "gbif",
            "query_scientific_name": record.get("species"),
            "matched_name": record.get("scientificName"),
            "source_observation_id": record.get("key"),
            "source_photo_id": None,
            "url": image_url,
            "raw_url": image_url,
            "license": m.get("license") or record.get("license"),
            "attribution": m.get("rightsHolder") or record.get("rightsHolder"),
            "reference": record.get("references") or record.get("occurrenceID"),
            "observed_on": record.get("eventDate"),
            "quality_grade": None,
            "latitude": record.get("decimalLatitude"),
            "longitude": record.get("decimalLongitude"),
            "gbif_dataset_key": record.get("datasetKey"),
            "gbif_publisher": record.get("publishingOrgKey"),
        })

    return rows


def query_gbif_species(gbif_species_id, scientific_name=None, max_photos=100, sleep_s=0.35):
    rows = []
    offset = 0
    limit = 300

    while len(rows) < max_photos:
        params = {
            "media_type": "StillImage",
            "limit": limit,
            "offset": offset,
        }

        # Prefer GBIF species key if available.
        if pd.notna(gbif_species_id):
            params["taxon_key"] = int(gbif_species_id)
        elif scientific_name:
            params["scientific_name"] = scientific_name
        else:
            break

        try:
            r = requests.get(GBIF_OCC_URL, params=params, timeout=30)
        except Exception as e:
            print("GBIF request exception:", gbif_species_id, scientific_name, repr(e))
            break

        if r.status_code != 200:
            print("GBIF error:", r.status_code, gbif_species_id, scientific_name, r.text[:300])
            break

        data = r.json()
        results = data.get("results", [])

        if not results:
            break

        for rec in results:
            media_rows = extract_gbif_media_rows(rec)

            for mr in media_rows:
                rows.append(mr)

                if len(rows) >= max_photos:
                    break

            if len(rows) >= max_photos:
                break

        if data.get("endOfRecords", False):
            break

        offset += limit
        time.sleep(sleep_s)

    return rows

## Collect candidates from iNat first, GBIF second

In [9]:
all_candidate_rows = []

for _, row in tqdm(targets.iterrows(), total=len(targets), desc="Collecting candidates"):
    species_id = row["species_id"]
    scientific_name = row["scientific_name"]
    gbif_species_id = row["gbif_species_id"]
    needed = int(row["needed_to_100"])

    max_candidates = min(MAX_CANDIDATES_PER_SPECIES, max(needed * CANDIDATE_MULTIPLIER, needed + 20))

    # 1. Try iNaturalist first.
    inat_rows = query_inat_species(
        scientific_name=scientific_name,
        max_photos=max_candidates,
        sleep_s=REQUEST_SLEEP_S,
    )

    # 2. If iNat does not find enough, use GBIF fallback.
    remaining = max_candidates - len(inat_rows)

    gbif_rows = []
    if remaining > 0:
        gbif_rows = query_gbif_species(
            gbif_species_id=gbif_species_id,
            scientific_name=scientific_name,
            max_photos=remaining,
            sleep_s=REQUEST_SLEEP_S,
        )

    rows = inat_rows + gbif_rows

    for r in rows:
        r["species_id"] = species_id
        r["gbif_species_id"] = gbif_species_id
        r["scientific_name"] = scientific_name
        r["current_n_images"] = row["n_images"]
        r["needed_to_100"] = needed

    all_candidate_rows.extend(rows)

candidates = pd.DataFrame(all_candidate_rows)

print("Candidate rows:", len(candidates))
candidates.head()

iNat error: 429 Limonium thiniense Erben <?xml version="1.0" encoding="utf-8"?>
<!DOCTYPE html PUBLIC "-//W3C//DTD XHTML 1.0 Strict//EN"
  "http://www.w3.org/TR/xhtml1/DTD/xhtml1-strict.dtd">
<html xmlns="http://www.w3.org/1999/xhtml" xml:lang="en" lang="en">
  <head>
    <title>429 Too Many Requests</title>
  </head>
  <body>
    <h1>Too 
iNat error: 429 Astragalus devesae Talavera & A.González & G.López <?xml version="1.0" encoding="utf-8"?>
<!DOCTYPE html PUBLIC "-//W3C//DTD XHTML 1.0 Strict//EN"
  "http://www.w3.org/TR/xhtml1/DTD/xhtml1-strict.dtd">
<html xmlns="http://www.w3.org/1999/xhtml" xml:lang="en" lang="en">
  <head>
    <title>429 Too Many Requests</title>
  </head>
  <body>
    <h1>Too 
iNat error: 429 Cirsium italicum DC. <?xml version="1.0" encoding="utf-8"?>
<!DOCTYPE html PUBLIC "-//W3C//DTD XHTML 1.0 Strict//EN"
  "http://www.w3.org/TR/xhtml1/DTD/xhtml1-strict.dtd">
<html xmlns="http://www.w3.org/1999/xhtml" xml:lang="en" lang="en">
  <head>
    <title>429 Too Many R

,api_source,query_scientific_name,matched_name,source_observation_id,source_photo_id,url,raw_url,license,attribution,reference,...,quality_grade,latitude,longitude,gbif_dataset_key,gbif_publisher,species_id,gbif_species_id,scientific_name,current_n_images,needed_to_100
0,gbif,Limonium subglabrum,Limonium subglabrum Erben,3859206900,None,https://inaturalist-open-data.s3.amazonaws.com...,https://inaturalist-open-data.s3.amazonaws.com...,http://creativecommons.org/licenses/by-nc/4.0/,thehousebunting,https://www.inaturalist.org/observations/10790...,...,None,37.102875,-3.721317,50c9509d-22c7-4a22-a47d-8c48425ef4a7,28eb1a3f-1c15-4a95-931a-4af90ecb574d,1744380,4089427.0,Limonium subglabrum Erben,1,99
1,gbif,Limonium subglabrum,Limonium subglabrum Erben,694543795,None,http://mediaphoto.mnhn.fr/media/14414219450736...,http://mediaphoto.mnhn.fr/media/14414219450736...,http://creativecommons.org/licenses/by/4.0/,NaN,http://coldb.mnhn.fr/catalognumber/mnhn/p/p050...,...,None,NaN,NaN,b5cdf794-8fa4-4a85-8b26-755d087bf531,2cd829bb-b713-433d-99cf-64bef11e5b3e,1744380,4089427.0,Limonium subglabrum Erben,1,99
2,gbif,Limonium subglabrum,Limonium subglabrum Erben,4508288536,None,https://archimg.mnhn.lu/Collections/Herbarium_...,https://archimg.mnhn.lu/Collections/Herbarium_...,http://creativecommons.org/publicdomain/zero/1...,NaN,DSS0043900002B3R,...,None,37.091100,-3.708500,962f59bc-f762-11e1-a439-00145eb45e9a,75642970-f855-11dd-8235-b8a03c50a862,1744380,4089427.0,Limonium subglabrum Erben,1,99
3,gbif,Limonium subglabrum,Limonium subglabrum Erben,1883453032,None,http://pictures.snsb.info/BSMvplantscoll/web/M...,http://pictures.snsb.info/BSMvplantscoll/web/M...,http://creativecommons.org/licenses/by-sa/4.0/,NaN,http://biocase.snsb.info/wrapper/querytool/det...,...,None,NaN,NaN,7b9a08ea-f762-11e1-a439-00145eb45e9a,0674aea0-a7e1-11d8-9534-b8a03c50a862,1744380,4089427.0,Limonium subglabrum Erben,1,99
4,gbif,Limonium subglabrum,Limonium subglabrum Erben,1936014699,None,https://assets.rjb.csic.es/fileget?coll=Planta...,https://assets.rjb.csic.es/fileget?coll=Planta...,http://creativecommons.org/licenses/by-nc/4.0/,NaN,34d90f36-20fe-11e3-9e2e-000c29035824,...,None,NaN,NaN,834c9918-f762-11e1-a439-00145eb45e9a,0363cbd4-f666-455e-8e86-0bbddcf51950,1744380,4089427.0,Limonium subglabrum Erben,1,99


Save raw candidates:

In [10]:
raw_candidates_out = OUT_DIR / "under100_inat_gbif_candidates_raw.csv"
candidates.to_csv(raw_candidates_out, index=False)

print("Saved:", raw_candidates_out)

Saved: /root/workspace/PlantCLEF2026/src_experiments/i001_data_download/data/extra_under100/under100_inat_gbif_candidates_raw.csv


## Clean candidates

In [12]:
candidates_clean = candidates.copy()

candidates_clean = candidates_clean.dropna(subset=["url"]).copy()

# Normalize license strings
candidates_clean["license"] = candidates_clean["license"].astype(str).str.strip().str.lower()

before = len(candidates_clean)
candidates_clean = candidates_clean.drop_duplicates(subset=["url"]).copy()
after = len(candidates_clean)

print("Removed duplicate URLs:", before - after)
print("Remaining candidates:", len(candidates_clean))

candidates_clean["api_source"].value_counts(dropna=False), candidates_clean["license"].value_counts(dropna=False).head(20)

Removed duplicate URLs: 1097
Remaining candidates: 276825


(api_source
 gbif    276825
 Name: count, dtype: int64,
 license
 http://creativecommons.org/licenses/by/4.0/                                                                                                                                                                                                                                                                                                                                                                       150913
 http://creativecommons.org/licenses/by-nc/4.0/                                                                                                                                                                                                                                                                                                                                                                     54537
 http://creativecommons.org/licenses/by-nc-nd/4.0/                                                                 

In [13]:
if ALLOW_NC:
    allowed_license_fragments = [
        "cc0",
        "cc-by",
        "cc by",
        "creativecommons.org/licenses/by/",
        "creativecommons.org/licenses/by-sa/",
        "creativecommons.org/licenses/by-nc/",
        "creativecommons.org/licenses/by-nc-sa/",
        "creativecommons.org/publicdomain/zero/",
    ]
else:
    allowed_license_fragments = [
        "cc0",
        "cc-by",
        "cc by",
        "creativecommons.org/licenses/by/",
        "creativecommons.org/licenses/by-sa/",
        "creativecommons.org/publicdomain/zero/",
    ]


def license_allowed(x):
    x = str(x).lower()
    return any(fragment in x for fragment in allowed_license_fragments)


before = len(candidates_clean)
candidates_clean = candidates_clean[candidates_clean["license"].apply(license_allowed)].copy()
after = len(candidates_clean)

print("Removed by license filter:", before - after)
print("Remaining:", after)

candidates_clean["license"].value_counts(dropna=False).head(30)

Removed by license filter: 19585
Remaining: 257240


license
http://creativecommons.org/licenses/by/4.0/                                                                                                                                                                                                                                                                                                                                                                       150913
http://creativecommons.org/licenses/by-nc/4.0/                                                                                                                                                                                                                                                                                                                                                                     54537
http://creativecommons.org/publicdomain/zero/1.0/                                                                                                                             

Save clean candidates:

In [14]:
clean_candidates_out = OUT_DIR / "under100_inat_gbif_candidates_clean.csv"
candidates_clean.to_csv(clean_candidates_out, index=False)

print("Saved:", clean_candidates_out)

Saved: /root/workspace/PlantCLEF2026/src_experiments/i001_data_download/data/extra_under100/under100_inat_gbif_candidates_clean.csv


## Select enough candidates per species

In [15]:
selected_parts = []

for species_id, group in candidates_clean.groupby("species_id"):
    needed = int(group["needed_to_100"].iloc[0])
    n_select = min(len(group), int(needed * 1.5) + 10)

    # Prefer iNat over GBIF when available, then keep stable order.
    group = group.copy()
    group["source_priority"] = group["api_source"].map({"inat": 0, "gbif": 1}).fillna(9)
    group = group.sort_values(["source_priority"])

    selected_parts.append(group.head(n_select))

selected_candidates = pd.concat(selected_parts, ignore_index=True) if selected_parts else pd.DataFrame()

print("Selected candidates:", len(selected_candidates))
print("Species covered:", selected_candidates["species_id"].nunique())

selected_candidates.head()

Selected candidates: 180964
Species covered: 2029


,api_source,query_scientific_name,matched_name,source_observation_id,source_photo_id,url,raw_url,license,attribution,reference,...,latitude,longitude,gbif_dataset_key,gbif_publisher,species_id,gbif_species_id,scientific_name,current_n_images,needed_to_100,source_priority
0,gbif,Arctotis venusta,Arctotis venusta Norl.,6159402929,None,https://inaturalist-open-data.s3.amazonaws.com...,https://inaturalist-open-data.s3.amazonaws.com...,http://creativecommons.org/licenses/by-nc/4.0/,Silke Rugheimer,https://www.inaturalist.org/observations/33834...,...,-22.311819,17.357323,50c9509d-22c7-4a22-a47d-8c48425ef4a7,28eb1a3f-1c15-4a95-931a-4af90ecb574d,1355873,5406166.0,Arctotis venusta Norl.,98,2,1
1,gbif,Arctotis venusta,Arctotis venusta Norl.,6159402929,None,https://inaturalist-open-data.s3.amazonaws.com...,https://inaturalist-open-data.s3.amazonaws.com...,http://creativecommons.org/licenses/by-nc/4.0/,Silke Rugheimer,https://www.inaturalist.org/observations/33834...,...,-22.311819,17.357323,50c9509d-22c7-4a22-a47d-8c48425ef4a7,28eb1a3f-1c15-4a95-931a-4af90ecb574d,1355873,5406166.0,Arctotis venusta Norl.,98,2,1
2,gbif,Arctotis venusta,Arctotis venusta Norl.,6196028337,None,https://inaturalist-open-data.s3.amazonaws.com...,https://inaturalist-open-data.s3.amazonaws.com...,http://creativecommons.org/licenses/by-nc/4.0/,Lize Koolmees,https://www.inaturalist.org/observations/33537...,...,-29.251096,27.378486,50c9509d-22c7-4a22-a47d-8c48425ef4a7,28eb1a3f-1c15-4a95-931a-4af90ecb574d,1355873,5406166.0,Arctotis venusta Norl.,98,2,1
3,gbif,Arctotis venusta,Arctotis venusta Norl.,6196028337,None,https://inaturalist-open-data.s3.amazonaws.com...,https://inaturalist-open-data.s3.amazonaws.com...,http://creativecommons.org/licenses/by-nc/4.0/,Lize Koolmees,https://www.inaturalist.org/observations/33537...,...,-29.251096,27.378486,50c9509d-22c7-4a22-a47d-8c48425ef4a7,28eb1a3f-1c15-4a95-931a-4af90ecb574d,1355873,5406166.0,Arctotis venusta Norl.,98,2,1
4,gbif,Arctotis venusta,Arctotis venusta Norl.,6196028337,None,https://inaturalist-open-data.s3.amazonaws.com...,https://inaturalist-open-data.s3.amazonaws.com...,http://creativecommons.org/licenses/by-nc/4.0/,Lize Koolmees,https://www.inaturalist.org/observations/33537...,...,-29.251096,27.378486,50c9509d-22c7-4a22-a47d-8c48425ef4a7,28eb1a3f-1c15-4a95-931a-4af90ecb574d,1355873,5406166.0,Arctotis venusta Norl.,98,2,1


In [16]:
selected_out = OUT_DIR / "under100_selected_candidates_for_download.csv"
selected_candidates.to_csv(selected_out, index=False)

print("Saved:", selected_out)

Saved: /root/workspace/PlantCLEF2026/src_experiments/i001_data_download/data/extra_under100/under100_selected_candidates_for_download.csv


## Download images in parallel

In [17]:
def file_ext_from_url(url):
    clean = str(url).split("?")[0]
    ext = Path(clean).suffix.lower()
    if ext in [".jpg", ".jpeg", ".png", ".webp"]:
        return ext
    return ".jpg"


def stable_url_id(url):
    return hashlib.sha1(str(url).encode("utf-8")).hexdigest()[:16]


def download_one(row_dict):
    species_id = str(row_dict["species_id"])
    api_source = row_dict["api_source"]
    url = row_dict["url"]

    species_dir = DOWNLOAD_ROOT / api_source / species_id
    species_dir.mkdir(parents=True, exist_ok=True)

    ext = file_ext_from_url(url)
    uid = stable_url_id(url)
    out_path = species_dir / f"{api_source}_{uid}{ext}"

    if out_path.exists() and out_path.stat().st_size > 0:
        return {
            **row_dict,
            "image_path": str(out_path),
            "download_ok": True,
            "download_status": "exists",
        }

    try:
        headers = {
            "User-Agent": "PlantCLEF2026 research image collection script"
        }

        r = requests.get(url, timeout=40, stream=True, headers=headers)

        if r.status_code != 200:
            return {
                **row_dict,
                "image_path": str(out_path),
                "download_ok": False,
                "download_status": f"http_{r.status_code}",
            }

        with open(out_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)

        if not out_path.exists() or out_path.stat().st_size == 0:
            return {
                **row_dict,
                "image_path": str(out_path),
                "download_ok": False,
                "download_status": "empty_file",
            }

        return {
            **row_dict,
            "image_path": str(out_path),
            "download_ok": True,
            "download_status": "downloaded",
        }

    except Exception as e:
        return {
            **row_dict,
            "image_path": str(out_path),
            "download_ok": False,
            "download_status": repr(e),
        }

In [18]:
rows = selected_candidates.to_dict("records")

download_results = []

MAX_WORKERS = 16

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
    futures = [ex.submit(download_one, r) for r in rows]

    for fut in tqdm(as_completed(futures), total=len(futures), desc="Downloading"):
        download_results.append(fut.result())

downloaded = pd.DataFrame(download_results)

print("Downloaded OK:", int(downloaded["download_ok"].sum()))
print("Failed:", int((~downloaded["download_ok"]).sum()))

downloaded.head()

Downloading:   0%|          | 0/180964 [00:00<?, ?it/s]

Downloaded OK: 161686
Failed: 19278


,api_source,query_scientific_name,matched_name,source_observation_id,source_photo_id,url,raw_url,license,attribution,reference,...,gbif_publisher,species_id,gbif_species_id,scientific_name,current_n_images,needed_to_100,source_priority,image_path,download_ok,download_status
0,gbif,Arctotis venusta,Arctotis venusta Norl.,6196028337,None,https://inaturalist-open-data.s3.amazonaws.com...,https://inaturalist-open-data.s3.amazonaws.com...,http://creativecommons.org/licenses/by-nc/4.0/,Lize Koolmees,https://www.inaturalist.org/observations/33537...,...,28eb1a3f-1c15-4a95-931a-4af90ecb574d,1355873,5406166.0,Arctotis venusta Norl.,98,2,1,/workspace/plantclef/raw/extra_under100/gbif/1...,True,downloaded
1,gbif,Arctotis venusta,Arctotis venusta Norl.,6195688758,None,https://inaturalist-open-data.s3.amazonaws.com...,https://inaturalist-open-data.s3.amazonaws.com...,http://creativecommons.org/licenses/by-nc/4.0/,Connor Ryan,https://www.inaturalist.org/observations/33729...,...,28eb1a3f-1c15-4a95-931a-4af90ecb574d,1355873,5406166.0,Arctotis venusta Norl.,98,2,1,/workspace/plantclef/raw/extra_under100/gbif/1...,True,downloaded
2,gbif,Arctotis venusta,Arctotis venusta Norl.,6159402929,None,https://inaturalist-open-data.s3.amazonaws.com...,https://inaturalist-open-data.s3.amazonaws.com...,http://creativecommons.org/licenses/by-nc/4.0/,Silke Rugheimer,https://www.inaturalist.org/observations/33834...,...,28eb1a3f-1c15-4a95-931a-4af90ecb574d,1355873,5406166.0,Arctotis venusta Norl.,98,2,1,/workspace/plantclef/raw/extra_under100/gbif/1...,True,downloaded
3,gbif,Arctotis venusta,Arctotis venusta Norl.,6159402929,None,https://inaturalist-open-data.s3.amazonaws.com...,https://inaturalist-open-data.s3.amazonaws.com...,http://creativecommons.org/licenses/by-nc/4.0/,Silke Rugheimer,https://www.inaturalist.org/observations/33834...,...,28eb1a3f-1c15-4a95-931a-4af90ecb574d,1355873,5406166.0,Arctotis venusta Norl.,98,2,1,/workspace/plantclef/raw/extra_under100/gbif/1...,True,downloaded
4,gbif,Arctotis venusta,Arctotis venusta Norl.,6195688758,None,https://inaturalist-open-data.s3.amazonaws.com...,https://inaturalist-open-data.s3.amazonaws.com...,http://creativecommons.org/licenses/by-nc/4.0/,Connor Ryan,https://www.inaturalist.org/observations/33729...,...,28eb1a3f-1c15-4a95-931a-4af90ecb574d,1355873,5406166.0,Arctotis venusta Norl.,98,2,1,/workspace/plantclef/raw/extra_under100/gbif/1...,True,downloaded


Save download results:

In [19]:
downloaded_out = OUT_DIR / "under100_downloaded_results.csv"
downloaded.to_csv(downloaded_out, index=False)

print("Saved:", downloaded_out)

Saved: /root/workspace/PlantCLEF2026/src_experiments/i001_data_download/data/extra_under100/under100_downloaded_results.csv


## Create extra training manifest

In [20]:
extra = downloaded[downloaded["download_ok"]].copy()

extra_manifest = pd.DataFrame({
    "image_path": extra["image_path"],
    "image_name": extra["image_path"].astype(str).apply(lambda x: Path(x).name),
    "species_id": extra["species_id"],
    "gbif_species_id": extra["gbif_species_id"],
    "scientific_name": extra["scientific_name"],
    "license": extra["license"],
    "source": "extra_" + extra["api_source"].astype(str) + "_under100",
    "url": extra["url"],
    "reference": extra["reference"],
    "attribution": extra["attribution"],
    "api_source": extra["api_source"],
    "source_observation_id": extra["source_observation_id"],
    "source_photo_id": extra["source_photo_id"],
})

extra_manifest = extra_manifest.drop_duplicates(subset=["image_path"]).copy()

print("Extra images:", len(extra_manifest))
print("Extra species:", extra_manifest["species_id"].nunique())

extra_manifest.head()

Extra images: 161686
Extra species: 2022


,image_path,image_name,species_id,gbif_species_id,scientific_name,license,source,url,reference,attribution,api_source,source_observation_id,source_photo_id
0,/workspace/plantclef/raw/extra_under100/gbif/1...,gbif_05397287ebc09076.jpg,1355873,5406166.0,Arctotis venusta Norl.,http://creativecommons.org/licenses/by-nc/4.0/,extra_gbif_under100,https://inaturalist-open-data.s3.amazonaws.com...,https://www.inaturalist.org/observations/33537...,Lize Koolmees,gbif,6196028337,None
1,/workspace/plantclef/raw/extra_under100/gbif/1...,gbif_80572245f954f897.jpg,1355873,5406166.0,Arctotis venusta Norl.,http://creativecommons.org/licenses/by-nc/4.0/,extra_gbif_under100,https://inaturalist-open-data.s3.amazonaws.com...,https://www.inaturalist.org/observations/33729...,Connor Ryan,gbif,6195688758,None
2,/workspace/plantclef/raw/extra_under100/gbif/1...,gbif_8bb8c221cc67e12a.jpg,1355873,5406166.0,Arctotis venusta Norl.,http://creativecommons.org/licenses/by-nc/4.0/,extra_gbif_under100,https://inaturalist-open-data.s3.amazonaws.com...,https://www.inaturalist.org/observations/33834...,Silke Rugheimer,gbif,6159402929,None
3,/workspace/plantclef/raw/extra_under100/gbif/1...,gbif_01a5aff0a952b830.jpg,1355873,5406166.0,Arctotis venusta Norl.,http://creativecommons.org/licenses/by-nc/4.0/,extra_gbif_under100,https://inaturalist-open-data.s3.amazonaws.com...,https://www.inaturalist.org/observations/33834...,Silke Rugheimer,gbif,6159402929,None
4,/workspace/plantclef/raw/extra_under100/gbif/1...,gbif_2b80dad07f7d1264.jpg,1355873,5406166.0,Arctotis venusta Norl.,http://creativecommons.org/licenses/by-nc/4.0/,extra_gbif_under100,https://inaturalist-open-data.s3.amazonaws.com...,https://www.inaturalist.org/observations/33729...,Connor Ryan,gbif,6195688758,None


Save:

In [21]:
extra_manifest_out = OUT_DIR / "extra_under100_train_manifest.csv"
extra_manifest.to_csv(extra_manifest_out, index=False)

print("Saved:", extra_manifest_out)

Saved: /root/workspace/PlantCLEF2026/src_experiments/i001_data_download/data/extra_under100/extra_under100_train_manifest.csv
